# 01: 数据读入与标准化

通过 `read_with_manifest` 读入原始单细胞数据并自动完成标准化步骤，写入 01 checkpoint `.h5ad` 文件。

**为什么把数据读入放在独立的 01？** 多来源数据的格式、obs 列名、基因 ID 体系各异。
把标准化集中在 01，后续所有分析直接操作统一的 AnnData，无需每步检查"这个数据集的列名是什么"。

**本 notebook 产出**：
- 跨来源数据集标准化的 `obs`（统一列名体系）
- 基线 QC 指标：`n_genes`、`total_counts`、`pct_counts_mt`、`pct_counts_ribo`
- `var.index` = gene symbol + `var["ensembl_id"]` = Ensembl ID
- 01 checkpoint `.h5ad` 文件，供 02 QC 使用

In [ ]:
# ============================================================
# PARAMS —— 运行前需编辑的参数集中在这里
# ============================================================

# MANIFEST_PATH: 数据集描述文件路径。
#   每个数据集一个 manifest.yaml，记录数据格式、来源、基因ID体系、
#   疾病/物种等元信息。切换数据集时改这里即可。
MANIFEST_PATH = "data/nancang/manifest.yaml"

# OUTPUT_PATH: 01 checkpoint 写入路径。
#   文件名不含数据集名前缀——合并后是多源整合对象，不再是单个数据集。
#   切换 MANIFEST_PATH 时记得同步更新此路径。
OUTPUT_PATH   = "results/01_loaded_v1.h5ad"

# RANDOM_SEED: 固定随机种子，保证可复现。
RANDOM_SEED   = 42

In [ ]:
# ============================================================
# 环境设置 —— 确保框架 src/ 可导入，然后一次性加载所有依赖
# ============================================================
# sys.path 必须在框架 import 之前设置，否则 nbconvert 干净 kernel 下
# "from scrna_integration import ..." 会因找不到模块而崩溃。
import sys, os

_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
print(f"PROJECT_ROOT: {_root}")

# --- 所有 import 集中在这里 ---
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import yaml
import warnings
import matplotlib.pyplot as plt
import importlib.metadata

# 框架入口：唯一 IO 入口函数
# 注意：此行必须在 sys.path 设置之后，否则 import 失败
from scrna_integration import read_with_manifest

# 抑制 mygene / scanpy 在读取过程中的冗余警告
warnings.filterwarnings("ignore", message=".*mygene.*")
warnings.filterwarnings("ignore", message=".*Layer2.*")

sc.settings.verbosity = 2  # 显示有用进度（0=quiet, 3=verbose）

print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

## 第一步：读入数据

`read_with_manifest` 是框架唯一的 IO 入口函数。它接收一个 manifest.yaml 路径，
内部自动完成：格式统一（10x / h5ad / loom → AnnData）、obs 列名标准化、
基因 ID 双向同步（gene symbol ↔ Ensembl）、基线 QC 指标计算。

**看什么**：读入后先确认细胞数和基因数是否符合预期、obs 列是否正确填充。

In [ ]:
# 调用 read_with_manifest——框架唯一的 IO 入口。
print(f"\n===== 正在从 {MANIFEST_PATH} 读入 =====\n")
adata = read_with_manifest(MANIFEST_PATH)

print(f"\n返回 AnnData: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

## 第二步：数据概览——obs / var / uns

- **`adata.obs`**：细胞级别元数据表，每行一个细胞。关键的标准化列包括
  `source_dataset`（来源数据集）、`sample_id`（样本 ID）。
- **`adata.var`**：基因级别元数据表。`var.index` 是 gene symbol，
  `var["ensembl_id"]` 是 Ensembl ID。
- **`adata.uns`**：附加字典，包含原始数据路径、stage 标记等。

**看什么**：obs 列名是否齐全、关键列是否正确填充、var 是否同时有 gene symbol 和 Ensembl ID。

In [ ]:
# 查看 obs 表前几行——每个细胞一行，列是标准化后的元数据
print("=== obs 前 5 行 ===")
display(adata.obs.head())

print("\n=== var 前 5 行 ===")
display(adata.var.head())

print("\n=== obs 列清单 ===")
print(list(adata.obs.columns))

print("\n=== uns 键 ===")
print(list(adata.uns.keys()))

# stage 记录——用于追踪当前数据的处理阶段
print(f"\nstage: {adata.uns.get('stage', '未设置')}")

## 图1：样本细胞构成

**看什么**：每个样本贡献了多少细胞。样本间细胞数差异过大可能提示建库质量差异
或实验批次效应，后续分析需要留意。当数据包含多个来源时，会额外画出按来源的分布。

In [ ]:
# 按样本统计细胞数，画条形图
sample_counts = adata.obs['sample_id'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(sample_counts)), sample_counts.values,
              color='steelblue', edgecolor='white')
ax.set_xticks(range(len(sample_counts)))
ax.set_xticklabels(sample_counts.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('细胞数')
ax.set_title(f'各样本细胞数（共 {adata.n_obs:,} 细胞，{len(sample_counts)} 个样本）')

# 在柱子上标注数值
for bar, val in zip(bars, sample_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

# 如果来源不止一个数据集，额外画按 source_dataset 分组的细胞数
n_sources = adata.obs['source_dataset'].nunique()
if n_sources > 1:
    source_counts = adata.obs['source_dataset'].value_counts()
    fig, ax = plt.subplots(figsize=(max(6, n_sources * 2), 4))
    source_counts.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
    ax.set_ylabel('细胞数')
    ax.set_title(f'各来源数据集细胞数（{n_sources} 个来源）')
    plt.tight_layout()
    plt.show()
else:
    print(
        f"来源数据集数: {n_sources}"
        f"（{list(adata.obs['source_dataset'].unique())}），"
        f"单源数据，不额外画按来源分布。"
    )

## 图2：元数据列填充情况与样本交叉构成

**看什么**：哪些元数据列有实际值（非空）？哪些完全缺失？如果 `disease`、`sex`、
`tissue` 等列全部为 NaN，说明原始数据集未提供这些信息——后续可从外部 metadata 补入。

同时展示来源 × 样本的交叉构成：多源场景下判断不同来源的样本是否重叠，
单源时以表格展示各样本细胞分布。

In [ ]:
# 检查标准元数据列（Layer 1 核心列）的填充情况，并画条形图
metadata_cols = ['disease', 'disease_system', 'tissue', 'sex',
                 'development_stage', 'assay']
fill_info = {}
print("===== 标准元数据列填充情况 =====")
for col in metadata_cols:
    if col in adata.obs.columns:
        n_valid = adata.obs[col].notna().sum()
        pct = n_valid / adata.n_obs * 100
        fill_info[col] = pct
        if n_valid > 0:
            vals = adata.obs[col].dropna().unique()
            print(f"  {col:25s}: {n_valid}/{adata.n_obs} ({pct:.0f}%)  取值: {list(vals[:5])}")
        else:
            print(f"  {col:25s}: 全空（数据集未提供此信息）")
    else:
        print(f"  {col:25s}: 列不存在")
        fill_info[col] = 0.0

# 画元数据列填充率条形图——一眼看出哪些列有数据
fig, ax = plt.subplots(figsize=(8, 4))
cols = list(fill_info.keys())
pcts = list(fill_info.values())
colors = ['steelblue' if p > 0 else 'lightgray' for p in pcts]
bars = ax.barh(range(len(cols)), pcts, color=colors, edgecolor='white')
ax.set_yticks(range(len(cols)))
ax.set_yticklabels(cols)
ax.set_xlabel('填充率 (%)')
ax.set_title('标准元数据列填充情况')
ax.invert_yaxis()  # 第一行在顶部
for bar, pct in zip(bars, pcts):
    label = f'{pct:.0f}%' if pct > 0 else '空'
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=9)
ax.set_xlim(0, 115)
plt.tight_layout()
plt.show()

# 来源 x 样本交叉表
print("\n===== 来源 × 样本 细胞数 =====")
cross_tab = pd.crosstab(adata.obs['source_dataset'], adata.obs['sample_id'])

if cross_tab.shape[0] == 1:
    # 单源数据：简洁表格
    display(pd.DataFrame({
        '样本': cross_tab.columns,
        '细胞数': [int(cross_tab.iloc[0, i]) for i in range(cross_tab.shape[1])]
    }).set_index('样本'))
else:
    # 多源数据：堆叠条形图
    cross_tab.T.plot(kind='bar', stacked=True, figsize=(12, 5),
                     colormap='Set2', edgecolor='white')
    plt.ylabel('细胞数')
    plt.title('各来源数据集 × 样本 细胞构成')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='source_dataset', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 图3：obs 列全览

**看什么**：这张表列出所有 obs 列及其填充情况（非空数、唯一值数、数据类型）。
全空列为后续 metadata 补入提供 TODO 清单。

In [ ]:
# obs 列信息总览——非空数 + 唯一值数 + 数据类型
obs_info = pd.DataFrame({
    '列名': adata.obs.columns,
    '非空数': [adata.obs[c].notna().sum() for c in adata.obs.columns],
    '唯一值数': [adata.obs[c].nunique() for c in adata.obs.columns],
    '数据类型': [str(adata.obs[c].dtype) for c in adata.obs.columns],
})
# 标注全空列
obs_info['状态'] = obs_info['非空数'].apply(lambda x: '有数据' if x > 0 else '全空')
display(obs_info)

# 有数据的 key 分类列，展示取值分布
print("\n--- 关键分类列取值分布 ---")
key_cols = [c for c in ['source_dataset', 'sample_id', 'disease_system']
            if c in adata.obs.columns and adata.obs[c].notna().any()]
for col in key_cols:
    print(f"\n{col}:")
    display(adata.obs[col].value_counts())

## 图4：基因空间概览

**看什么**：基因总数、含 Ensembl ID 的比例。多来源整合时，不同数据集的基因集合
可能不同（如 Nowicki ~25k vs Yue ~38k），基因交集大小影响跨数据集可比性。
单来源时仅展示基因数量统计。

In [ ]:
# 基因空间基础统计
print(f"基因总数: {adata.n_vars:,}")
if 'ensembl_id' in adata.var.columns:
    n_ensembl = adata.var['ensembl_id'].notna().sum()
    print(
        f"含 Ensembl ID 的基因: {n_ensembl:,} / {adata.n_vars:,}"
        f" ({n_ensembl/adata.n_vars*100:.1f}%)"
    )
else:
    print("var 中无 ensembl_id 列")

# 简单条形图
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['基因总数'], [adata.n_vars], color='mediumseagreen',
       edgecolor='white', width=0.4)
ax.set_ylabel('基因数')
ax.set_title('var 空间基因数')
ax.text(0, adata.n_vars + 100, str(adata.n_vars),
        ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# 单源提示
n_sources = adata.obs['source_dataset'].nunique()
if n_sources == 1:
    print("当前为单来源数据集，无跨源基因交集计算。多源整合时会自动处理基因交集。")

## 第三步：基线 QC 指标摘要

`read_with_manifest` 在原始 counts 上自动计算了 4 个基线 QC 指标：

| 指标 | 含义 | 用途 |
|------|------|------|
| `n_genes` | 每个细胞检测到的基因数 | 过低 → 死细胞/空液滴，过高 → 双细胞 |
| `total_counts` | 每个细胞的总 UMI 计数 | 反映测序深度 |
| `pct_counts_mt` | 线粒体基因 UMI 占比 | 过高 → 受损/濒死细胞 |
| `pct_counts_ribo` | 核糖体基因 UMI 占比 | 反映翻译活性 |

**看什么**：这四个指标的均值/中位/范围是否在合理区间。异常值（如 pct_mt > 50%）
提示数据质量问题，将在 02 QC 中处理。

In [ ]:
# 基线 QC 摘要
print("===== 基线 QC 摘要（原始 counts）=====")
for col in ["n_genes", "total_counts", "pct_counts_mt", "pct_counts_ribo"]:
    if col in adata.obs.columns:
        vals = adata.obs[col]
        print(
            f"  {col:20s}: 均值={vals.mean():.1f}  "
            f"中位={vals.median():.1f}  "
            f"最小={vals.min():.1f}  "
            f"最大={vals.max():.1f}"
        )

print(f"\n  来源数据集: {sorted(adata.obs['source_dataset'].unique())}")
n_samples = (
    adata.obs['sample_id'].nunique()
    if "sample_id" in adata.obs.columns
    else "N/A"
)
print(f"  样本数:      {n_samples}")

# raw_matrix_path（供 02 SoupX 环境 RNA 校正使用）
rp = adata.uns.get("raw_matrix_path", None)
print(f"\n  raw_matrix_path（供 SoupX 用）: {rp}")

# --- 写入前数据完整性检查 ---
# 确保 X 是稀疏矩阵且保持 float32——如果这里不是稀疏 float32，
# 说明上游某步不当 densify 或改了 dtype，会导致后续步骤内存溢出。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量违反: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("\n写入前检查通过: X 是 sparse CSR float32。")

## 第四步：写入 checkpoint 并释放内存

将当前 AnnData 写入磁盘形成 01 checkpoint，后续 notebook 可直接从磁盘加载，
无需重新执行读入步骤。写入后释放内存，避免在同一内核会话中运行后续 notebook 时
造成内存叠加。

In [ ]:
# 写入 stage checkpoint 到磁盘
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写入 {OUTPUT_PATH}")

# 校验文件已写入
assert os.path.exists(OUTPUT_PATH), f"输出未找到: {OUTPUT_PATH}"
print(f"已校验: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

# 释放 AnnData 对象——在同一内核会话中继续跑后续 notebook 时，
# 如果不释放，两个 stage 的 AnnData 会同时占用内存。
del adata
import gc
gc.collect()
print("内存已释放。")